In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import tensorflow as tf
from keras.applications import MobileNetV2

THU_MUC_GOC = Path("..").resolve()
sys.path.append(str(THU_MUC_GOC))

from data.preprocessor.bo_loc_anh import bo_loc

DATA_DA_XU_LY = THU_MUC_GOC / "data" / "processed"
THU_MUC_DAC_TRUNG = THU_MUC_GOC / "data" / "features"
THU_MUC_MODEL = THU_MUC_GOC / "model"

THU_MUC_DAC_TRUNG.mkdir(parents=True, exist_ok=True)
THU_MUC_MODEL.mkdir(parents=True, exist_ok=True)

with open(THU_MUC_MODEL / "labels.json", "r", encoding="utf-8") as f:
    label_map = json.load(f) 

ten_label = {ten : int(idx) for idx,ten in label_map.items()}
print("Nhãn:",ten_label)
print("Thiết bị đang chạy:", tf.config.list_physical_devices("GPU") or "CPU")

Nhãn: {'Huu_Co': 0, 'Tai_Che': 1, 'Vo_Co': 2}
Thiết bị đang chạy: CPU


In [18]:
bo_trich_xuat = MobileNetV2(
    weights = "imagenet",
    include_top = False,
    input_shape = (224,224,3),
    pooling="avg"
)
bo_trich_xuat.trainable = False

duong_dan_bo_trich_xuat = THU_MUC_MODEL / "bo_trich_xuat_mobilenetv2.keras"
bo_trich_xuat.save(duong_dan_bo_trich_xuat)
print("Kích thước đầu ra:",bo_trich_xuat.output_shape)
print("Đã lưu ở:",duong_dan_bo_trich_xuat)

Kích thước đầu ra: (None, 1280)
Đã lưu ở: C:\Users\Nguyen Vu Thuan\Documents\ML_bao_cao\model\bo_trich_xuat_mobilenetv2.keras


In [19]:
BATCH_SIZE = 64
def trich_xuat_tap(ten_tap:str):
    tap_thu_muc = DATA_DA_XU_LY / ten_tap
    danh_sach_anh =[]
    danh_sach_nhan=[]

    for ten_lop, idx_lop in ten_label.items():
        thu_muc_lop = tap_thu_muc / ten_lop
        if not thu_muc_lop.exists():
            continue
        for file_anh in sorted(thu_muc_lop.iterdir()):
            if file_anh.is_file():
                danh_sach_anh.append(file_anh)
                danh_sach_nhan.append(idx_lop)

    tong_anh = len(danh_sach_anh)
    print(f"\nBắt đầu trích xuất tập '{ten_tap}': {tong_anh} ảnh")

    dac_trung_list = []
    bat_dau = time.time()
    for i in range(0, tong_anh, BATCH_SIZE):
        lo_anh = danh_sach_anh[i : i + BATCH_SIZE]
        batch_tensor = np.stack([bo_loc(p) for p in lo_anh], axis=0)
        batch_features = np.asarray(bo_trich_xuat(batch_tensor, training=False), dtype=np.float32)
        dac_trung_list.append(batch_features)

        da_xong = min(i + BATCH_SIZE, tong_anh)
        if (i // BATCH_SIZE) % 20 == 0 or da_xong == tong_anh:
            thoi_gian = time.time() - bat_dau
            print(f"  Đã xử lý {da_xong}/{tong_anh} ảnh ({da_xong / tong_anh * 100:.1f}%) - {thoi_gian:.1f}s")

    X = np.vstack(dac_trung_list).astype(np.float32)
    y = np.array(danh_sach_nhan, dtype=np.int32)
    return X, y





In [20]:
X_train, y_train = trich_xuat_tap("train")
X_test, y_test = trich_xuat_tap("test")

assert not np.isnan(X_train).any() and not np.isinf(X_train).any(), "X_train chứa NaN hoặc Inf!"
assert not np.isnan(X_test).any() and not np.isinf(X_test).any(), "X_test chứa NaN hoặc Inf!"

np.save(THU_MUC_DAC_TRUNG / "X_train.npy", X_train)
np.save(THU_MUC_DAC_TRUNG / "y_train.npy", y_train)
np.save(THU_MUC_DAC_TRUNG / "X_test.npy", X_test)
np.save(THU_MUC_DAC_TRUNG / "y_test.npy", y_test)

print("\nHoàn tất lưu đặc trưng tại:", THU_MUC_DAC_TRUNG)
print(f"X_train: {X_train.shape} ({X_train.nbytes / 1e6:.1f} MB) | y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape} ({X_test.nbytes / 1e6:.1f} MB) | y_test:  {y_test.shape}")


Bắt đầu trích xuất tập 'train': 12888 ảnh
  Đã xử lý 64/12888 ảnh (0.5%) - 2.8s
  Đã xử lý 1344/12888 ảnh (10.4%) - 45.8s
  Đã xử lý 2624/12888 ảnh (20.4%) - 86.1s
  Đã xử lý 3904/12888 ảnh (30.3%) - 121.9s
  Đã xử lý 5184/12888 ảnh (40.2%) - 161.5s
  Đã xử lý 6464/12888 ảnh (50.2%) - 201.6s
  Đã xử lý 7744/12888 ảnh (60.1%) - 249.5s
  Đã xử lý 9024/12888 ảnh (70.0%) - 294.5s
  Đã xử lý 10304/12888 ảnh (80.0%) - 333.7s
  Đã xử lý 11584/12888 ảnh (89.9%) - 373.2s
  Đã xử lý 12864/12888 ảnh (99.8%) - 415.6s
  Đã xử lý 12888/12888 ảnh (100.0%) - 416.5s

Bắt đầu trích xuất tập 'test': 3222 ảnh
  Đã xử lý 64/3222 ảnh (2.0%) - 2.0s
  Đã xử lý 1344/3222 ảnh (41.7%) - 45.4s
  Đã xử lý 2624/3222 ảnh (81.4%) - 88.9s
  Đã xử lý 3222/3222 ảnh (100.0%) - 107.4s

Hoàn tất lưu đặc trưng tại: C:\Users\Nguyen Vu Thuan\Documents\ML_bao_cao\features
X_train: (12888, 1280) (66.0 MB) | y_train: (12888,)
X_test:  (3222, 1280) (16.5 MB) | y_test:  (3222,)
